# 원하는 포즈로 이미지 만드는 도구 (FLUX.2-klein 4B + OpenPose)

참조 사진 한 장의 **사람 자세**를 그대로 따라 하는 새 이미지를 만드는 튜토리얼 노트입니다.
교안의 「FLUX.2-klein 4B로 마음껏 이미지 만들기」 노트(Colab + ComfyUI + GGUF)를 출발점으로, 그 위에 **OpenPose 포즈 조건**을 얹었습니다.

```
참조 사진 ──▶ OpenPose ──▶ 관절 뼈대 그림 ──┐
                                            ├──▶ FLUX.2-klein 4B ──▶ 같은 자세의 새 이미지
프롬프트(누가, 어디서, 어떤 결로) ──────────┘
```

## 왜 ControlNet 대신 '참조 이미지'로 넣나요?
- 포즈 조건은 보통 **ControlNet**(생성 모델 옆에 붙는 보조 신경망)으로 넣습니다.
- 그런데 FLUX.2에는 2026-07 현재 **BFL 공식 ControlNet이 없습니다.**
- 대신 FLUX.2-klein은 **참조 이미지(multi-reference editing)** 를 모델 안에서 바로 받습니다.
  그래서 이 노트는 OpenPose로 뽑은 **뼈대 그림을 참조 이미지로 넣고**, 프롬프트로 "이 뼈대와 같은 자세로 그려라"라고 지시합니다.
- ControlNet만큼 정밀하지 않을 수 있습니다. 그래서 맨 뒤 **부록 B**에 진짜 ControlNet(SD 1.5 + OpenPose ControlNet)으로 같은 일을 하는 셀도 넣어 두었습니다. 두 결과를 비교해 보는 것도 관찰거리입니다.

## 셀 순서 한눈에
| 단계 | 하는 일 |
|---|---|
| 1 | GPU 확인 |
| 2 | ComfyUI + GGUF 노드 + OpenPose 라이브러리 설치 |
| 3 | 모델 파일 3종 다운로드 (약 11GB) |
| 4 | ComfyUI 서버 실행 |
| 5 | 참조 사진 올리기 |
| 6 | **포즈 추출** (사진 → 뼈대 그림) |
| 7 | **이미지 생성 함수** 준비 (뼈대 + 프롬프트 → 이미지) |
| 8 | 한 장 만들어 보기 → `samples/` 에 **저장** |
| 9 | 실험 A: 같은 포즈, 다른 프롬프트 |
| 10 | 실험 B: 다른 포즈, 같은 프롬프트 |
| 11 | `prompts.md` 기록 저장 + 결과 내려받기 |
| 12 | 관찰 기록 (무엇을 바꿔 보았고, 어떻게 달라졌는가) |

## 준비
1. 상단 메뉴 **[런타임 → 런타임 유형 변경 → T4 GPU]** 선택 후 저장
2. 참조로 쓸 **사람 사진 2장 이상**을 준비합니다. 몸 전체(또는 상반신 전체)가 보이고, 자세가 서로 뚜렷이 다른 사진이 좋습니다.
   본인이 찍은 사진을 권합니다. 남의 얼굴 사진을 GitHub에 올리면 초상권 문제가 생길 수 있습니다.
3. 아래 셀을 **위에서부터 순서대로** 실행합니다.

## 라이선스
- FLUX.2-klein **4B**: Apache 2.0 (상업 이용 가능). 단, 결과물에 담긴 타인의 초상·저작물 권리까지 해결해 주지는 않습니다.
- 이름이 같은 FLUX.2-klein **9B**는 비상업 전용이니 바꿔 쓰지 마십시오.
- OpenPose 가중치(`lllyasviel/Annotators`)는 연구용으로 공개된 것입니다.

## [단계 1] GPU 확인

**이 셀이 하는 일:** 런타임에 GPU가 붙어 있는지, 이름과 VRAM이 얼마인지 출력합니다. 무료 티어는 보통 Tesla T4(약 15GB)입니다.
GPU가 없으면 오류를 내고 멈춥니다. 그때는 런타임 유형을 T4 GPU로 바꾸고 다시 실행하십시오.

In [ ]:
# [단계 1] GPU 가 붙어 있는지 확인
import torch

print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError("GPU가 없습니다. [런타임 → 런타임 유형 변경 → T4 GPU] 후 다시 실행하세요.")

## [단계 2] 설치: ComfyUI + GGUF 노드 + OpenPose

**이 셀이 하는 일:**
- **ComfyUI**: 이미지 생성 파이프라인을 노드로 조립해 돌리는 도구. 여기서는 화면 없이 **API 서버**로만 씁니다.
- **ComfyUI-GGUF**: 4비트로 양자화한 GGUF 모델 파일을 읽게 해 주는 플러그인. (16GB → 2.6GB)
- **controlnet_aux**: 사진에서 사람 관절을 찾아 뼈대 그림으로 그려 주는 **OpenPose** 전처리기가 들어 있는 라이브러리.

처음 한 번은 몇 분 걸립니다.

In [ ]:
# [단계 2] ComfyUI 본체 + GGUF 노드 + OpenPose 전처리 라이브러리 설치
import os

%cd /content
if not os.path.isdir("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git      # ComfyUI 본체

%cd /content/ComfyUI
!pip install -q -r requirements.txt                               # ComfyUI 의존성

%cd /content/ComfyUI/custom_nodes
if not os.path.isdir("ComfyUI-GGUF"):
    !git clone https://github.com/city96/ComfyUI-GGUF.git         # GGUF 로더 노드
%cd /content/ComfyUI
!pip install -q gguf                                              # GGUF 파싱 라이브러리

!pip install -q controlnet_aux                                    # OpenPose 전처리기

print("\n완료: ComfyUI + GGUF 노드 + controlnet_aux 설치")

## [단계 3] 모델 파일 3종 다운로드

**이 셀이 하는 일:** FLUX.2-klein 4B 파이프라인을 이루는 세 파일을 ComfyUI 표준 폴더에 받습니다. 교안 노트와 같은 파일입니다.

| 역할 | 파일 | 크기 |
|---|---|---|
| 디퓨전 모델 (그림을 그리는 본체) | `flux-2-klein-4b-Q4_K_M.gguf` | 약 2.6GB |
| 텍스트 인코더 (프롬프트 → 숫자) | `qwen_3_4b.safetensors` | 약 8GB |
| VAE (latent ↔ 픽셀 변환) | `flux2-vae.safetensors` | 약 330MB |

이미 받은 파일은 건너뜁니다. 런타임이 끊겨 다시 실행해도 처음부터 받지 않습니다.

In [ ]:
# [단계 3] 모델 파일 3종을 ComfyUI 표준 폴더에 다운로드
import os
%cd /content/ComfyUI

os.makedirs("models/diffusion_models", exist_ok=True)
os.makedirs("models/text_encoders",  exist_ok=True)
os.makedirs("models/vae",            exist_ok=True)

def 다운로드(url, 저장경로):
    # 이미 받았으면 건너뛰고, 아니면 wget 으로 받는다
    if os.path.exists(저장경로) and os.path.getsize(저장경로) > 1_000_000:
        print(f"  ↳ 이미 있음: {저장경로} ({os.path.getsize(저장경로)/1e9:.2f} GB)")
        return
    print("  ↳ 다운로드 중...")
    !wget -q --show-progress -O "{저장경로}" "{url}"

print("[1/3] 디퓨전 모델 (GGUF Q4, 약 2.6GB)")
다운로드("https://huggingface.co/unsloth/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_K_M.gguf",
        "models/diffusion_models/flux-2-klein-4b-Q4_K_M.gguf")

print("[2/3] 텍스트 인코더 (Qwen3-4B, 약 8GB)")
다운로드("https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors",
        "models/text_encoders/qwen_3_4b.safetensors")

print("[3/3] VAE (약 330MB)")
다운로드("https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/vae/flux2-vae.safetensors",
        "models/vae/flux2-vae.safetensors")

!ls -lh models/diffusion_models/*.gguf models/text_encoders/qwen_3_4b.safetensors models/vae/flux2-vae.safetensors

## [단계 4] ComfyUI 서버 실행

**이 셀이 하는 일:** ComfyUI를 백그라운드로 띄웁니다. 서버는 Colab 안의 `http://127.0.0.1:8188` 에서 API로 요청을 받습니다.
`완료` 가 출력되면 준비된 것입니다. 실패하면 로그 끝부분을 보여 줍니다.

In [ ]:
# [단계 4] ComfyUI 서버를 백그라운드로 실행하고 준비될 때까지 기다린다
import subprocess, os, time

%cd /content/ComfyUI
!pkill -f "main.py" 2>/dev/null
time.sleep(2)
os.system("nohup python main.py --listen 127.0.0.1 --port 8188 > /content/comfyui.log 2>&1 &")

print("ComfyUI 기동 대기 중...")
for _ in range(60):
    time.sleep(1)
    code_ = subprocess.run("curl -s -o /dev/null -w '%{http_code}' http://127.0.0.1:8188/",
                           shell=True, capture_output=True, text=True).stdout
    if code_ == "200":
        print("완료: ComfyUI 서버 준비 (http://127.0.0.1:8188)")
        break
else:
    print("서버가 안 뜹니다. 로그:")
    print(open("/content/comfyui.log").read()[-1500:])

## [단계 5] 참조 사진 올리기

**이 셀이 하는 일:** 자세를 따올 사람 사진을 올리고 `pose_01.png`, `pose_02.png` … 로 이름을 바꿔 `samples/` 에 저장합니다.
- 셀을 실행하면 **[파일 선택]** 버튼이 나옵니다. 사진을 **2장 이상** 한꺼번에 고르십시오.
- 올린 순서대로 번호가 붙습니다. 실험 B(다른 포즈 비교)에 쓰도록 **자세가 서로 다른 사진**을 고르십시오. (예: 한 손 들기 / 의자에 앉기 / 팔짱 끼고 서기)
- 사진이 너무 크면 긴 변을 1024px로 줄여 저장합니다.

> 사진 없이 도구만 먼저 시험하려면 `USE_SAMPLE_URL = True` 로 바꾸면 Hugging Face의 ControlNet 예제 사진 1장을 받아 씁니다.
> 그 사진은 타인의 초상이니 **GitHub에 올리는 samples/ 에는 본인 사진으로 다시 만든 결과**를 넣으십시오.

In [ ]:
# [단계 5] 참조 사진을 올리고 samples/pose_NN.png 로 저장
import os, io, urllib.request
from PIL import Image, ImageOps

USE_SAMPLE_URL = False     # True 면 업로드 대신 예제 사진 1장을 받아 씀 (시험용)
SAMPLE_URL = "https://huggingface.co/lllyasviel/control_v11p_sd15_openpose/resolve/main/images/input.png"

SAMPLES = "/content/pose-image-tool/samples"
os.makedirs(SAMPLES, exist_ok=True)

def 사진_정리(raw_bytes, max_side=1024):
    # 휴대폰 사진의 회전 정보(EXIF)를 반영하고, RGB 로 바꾸고, 긴 변을 max_side 로 줄인다
    img = ImageOps.exif_transpose(Image.open(io.BytesIO(raw_bytes))).convert("RGB")
    img.thumbnail((max_side, max_side))
    return img

if USE_SAMPLE_URL:
    raws = [urllib.request.urlopen(SAMPLE_URL).read()]
else:
    from google.colab import files
    uploaded = files.upload()                 # 파일 선택 창이 뜬다
    raws = list(uploaded.values())

POSE_FILES = []
for i, raw in enumerate(raws, start=1):
    path = f"{SAMPLES}/pose_{i:02d}.png"
    사진_정리(raw).save(path)
    POSE_FILES.append(path)
    print("저장:", path)

print(f"\n참조 사진 {len(POSE_FILES)}장 준비 완료")

## [단계 6] 포즈 추출 — 사진에서 관절 뼈대 뽑기

**이 셀이 하는 일:**
1. OpenPose 모델(`lllyasviel/Annotators`)을 불러옵니다.
2. 참조 사진마다 **목·어깨·팔꿈치·손목·골반·무릎·발목** 같은 관절 위치를 찾아, 색깔 선으로 이은 **뼈대 그림**(검은 배경)을 만듭니다.
3. `samples/pose_NN_skeleton.png` 로 저장하고, 원본과 나란히 보여 줍니다.

이 뼈대 그림이 다음 단계에서 **포즈 조건**으로 들어갑니다. 원본 사진의 얼굴·옷·배경은 버리고 **자세 정보만** 남는다는 점이 핵심입니다.

> 뼈대가 비어 있거나(검은 화면) 팔다리가 빠졌다면, 몸이 더 크게·온전히 나온 사진으로 바꾸십시오. 관절을 못 찾으면 생성도 자세를 따라갈 수 없습니다.

In [ ]:
# [단계 6] OpenPose 로 참조 사진에서 관절 뼈대 그림을 뽑아 저장
import matplotlib.pyplot as plt
from PIL import Image
from controlnet_aux import OpenposeDetector

openpose = OpenposeDetector.from_pretrained("lllyasviel/Annotators")   # OpenPose 가중치 (처음 한 번 다운로드)

def 포즈_추출(photo_path):
    # 사진 → 관절 뼈대 그림(PIL). 몸통+손 관절을 그린다. 크기는 원본 사진과 같게 맞춘다
    photo = Image.open(photo_path).convert("RGB")
    skel = openpose(photo, include_body=True, include_hand=True, include_face=False,
                    detect_resolution=512, image_resolution=max(photo.size))
    return skel.resize(photo.size)

SKELETONS = []
for p in POSE_FILES:
    skel = 포즈_추출(p)
    out = p.replace(".png", "_skeleton.png")
    skel.save(out)
    SKELETONS.append(out)

# 원본과 뼈대를 나란히 확인
fig, axes = plt.subplots(len(POSE_FILES), 2, figsize=(8, 4 * len(POSE_FILES)), squeeze=False)
for row, (p, s) in enumerate(zip(POSE_FILES, SKELETONS)):
    axes[row][0].imshow(Image.open(p)); axes[row][0].set_title(os.path.basename(p))
    axes[row][1].imshow(Image.open(s)); axes[row][1].set_title(os.path.basename(s))
    for a in axes[row]: a.axis("off")
plt.tight_layout(); plt.show()

## [단계 7] 이미지 생성 함수 — 뼈대 + 프롬프트 → 같은 자세의 이미지

**이 셀이 하는 일:** 생성에 쓸 함수 `포즈로_생성()` 을 정의합니다. (실행해도 아직 그림은 안 나옵니다.)

함수 안에서 일어나는 일:
1. 뼈대 그림을 생성 크기(가로·세로 16의 배수, 긴 변 1024)로 맞춰 ComfyUI의 `input/` 폴더에 복사합니다.
2. ComfyUI 워크플로(노드 그래프)를 조립합니다. 교안 노트의 그래프에 **세 노드**를 더했습니다.
   - `LoadImage` → 뼈대 그림을 읽고
   - `VAEEncode` → 그림을 latent(모델이 다루는 압축 표현)로 바꾸고
   - `ReferenceLatent` → 그 latent를 프롬프트 조건에 **참조 이미지로 붙입니다.** ← 여기가 '포즈 조건'
3. 프롬프트 뒤에 "뼈대와 같은 자세로, 뼈대 선은 그리지 말 것"이라는 **포즈 지시문**을 자동으로 붙입니다.
   네거티브 프롬프트에는 "뼈대, 색깔 선, 점" 등을 넣어 뼈대가 그대로 찍혀 나오는 것을 막습니다.
4. 요청을 보내고, 끝날 때까지 기다렸다가 결과 이미지를 받아 저장합니다.
5. 사용한 프롬프트·시드·설정을 `LOG` 에 쌓아 둡니다. (단계 11에서 `prompts.md` 로 저장)

생성 설정은 교안 노트와 같은 **4스텝, cfg 4.0, euler/simple** 입니다.

In [ ]:
# [단계 7] ComfyUI API 로 '뼈대 + 프롬프트 → 이미지' 를 만드는 함수
import requests, time, json, urllib.parse, shutil, os
from PIL import Image

SERVER = "http://127.0.0.1:8188"
COMFY_INPUT = "/content/ComfyUI/input"
os.makedirs(COMFY_INPUT, exist_ok=True)

STEPS, CFG = 4, 4.0          # 교안 노트와 같은 생성 설정
LOG = []                     # 생성 기록 (prompts.md 로 저장됨)

# 프롬프트 뒤에 자동으로 붙는 포즈 지시문 / 기본 네거티브
POSE_INSTRUCTION = (
    " The person must take exactly the same body pose as the stick-figure skeleton in the reference image: "
    "same head direction, same arm and hand positions, same leg positions, same body angle. "
    "Do not draw the skeleton itself: no colored lines, no dots, no black background."
)
BASE_NEGATIVE = "stick figure, skeleton, colored lines, keypoints, dots, black background, extra limbs, deformed hands"

def 생성_크기(skel_path, long_side=1024):
    # 뼈대 그림의 비율을 유지하면서 긴 변을 long_side 로, 가로·세로를 16의 배수로 맞춘다
    w, h = Image.open(skel_path).size
    s = long_side / max(w, h)
    return max(16, round(w * s / 16) * 16), max(16, round(h * s / 16) * 16)

def 포즈로_생성(skel_path, prompt, out_path, seed=42, negative="", use_pose=True):
    # skel_path: 단계 6에서 만든 뼈대 그림 / prompt: 그릴 내용 / out_path: 저장 위치
    # use_pose=False 면 포즈 조건 없이 프롬프트만으로 생성 (비교용)
    W, H = 생성_크기(skel_path)

    # ① 뼈대 그림을 생성 크기에 맞춰 ComfyUI input/ 폴더에 둔다
    ref_name = "ref_" + os.path.basename(skel_path)
    Image.open(skel_path).convert("RGB").resize((W, H)).save(f"{COMFY_INPUT}/{ref_name}")

    full_prompt = prompt + (POSE_INSTRUCTION if use_pose else "")
    full_negative = ", ".join(x for x in [negative, BASE_NEGATIVE if use_pose else ""] if x)

    # ② 워크플로 조립 (교안 노트의 그래프 + 포즈 참조 노드 3개)
    wf = {
      "10": {"class_type": "UnetLoaderGGUF",  "inputs": {"unet_name": "flux-2-klein-4b-Q4_K_M.gguf"}},
      "8":  {"class_type": "CLIPLoader",      "inputs": {"clip_name": "qwen_3_4b.safetensors", "type": "flux2"}},
      "4":  {"class_type": "VAELoader",       "inputs": {"vae_name": "flux2-vae.safetensors"}},
      "6":  {"class_type": "CLIPTextEncode",  "inputs": {"text": full_prompt,  "clip": ["8", 0]}},   # 프롬프트
      "7":  {"class_type": "CLIPTextEncode",  "inputs": {"text": full_negative, "clip": ["8", 0]}},  # 네거티브
      "5":  {"class_type": "EmptyLatentImage", "inputs": {"width": W, "height": H, "batch_size": 1}},
      "9":  {"class_type": "VAEDecode",       "inputs": {"samples": ["3", 0], "vae": ["4", 0]}},
      "11": {"class_type": "SaveImage",       "inputs": {"images": ["9", 0], "filename_prefix": "pose_tool"}},
    }
    pos, neg = ["6", 0], ["7", 0]
    if use_pose:
        wf["12"] = {"class_type": "LoadImage",      "inputs": {"image": ref_name}}                          # 뼈대 그림 읽기
        wf["13"] = {"class_type": "VAEEncode",      "inputs": {"pixels": ["12", 0], "vae": ["4", 0]}}      # 그림 → latent
        wf["14"] = {"class_type": "ReferenceLatent", "inputs": {"conditioning": ["6", 0], "latent": ["13", 0]}}  # 프롬프트에 참조 붙이기
        wf["15"] = {"class_type": "ReferenceLatent", "inputs": {"conditioning": ["7", 0], "latent": ["13", 0]}}  # 네거티브에도 같은 참조
        pos, neg = ["14", 0], ["15", 0]
    wf["3"] = {"class_type": "KSampler", "inputs": {
        "seed": int(seed), "steps": STEPS, "cfg": CFG, "sampler_name": "euler", "scheduler": "simple",
        "denoise": 1.0, "model": ["10", 0], "positive": pos, "negative": neg, "latent_image": ["5", 0]}}

    # ③ 요청 보내기
    r = requests.post(f"{SERVER}/prompt", json={"prompt": wf, "client_id": "pose-tool"}, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"요청 실패 {r.status_code}: {r.text[:800]}")
    pid = r.json()["prompt_id"]

    # ④ 끝날 때까지 기다렸다가 결과 받기
    t0 = time.time()
    for _ in range(300):
        time.sleep(1)
        h = requests.get(f"{SERVER}/history/{pid}", timeout=10).json()
        if pid in h:
            entry = h[pid]
            if entry.get("status", {}).get("status_str") == "error":
                raise RuntimeError("ComfyUI 실행 에러: " + json.dumps(entry["status"].get("messages", []), ensure_ascii=False)[:1500])
            img = entry["outputs"]["11"]["images"][0]
            break
    else:
        raise TimeoutError("5분 안에 끝나지 않았습니다. /content/comfyui.log 를 확인하세요.")

    q = urllib.parse.urlencode({"filename": img["filename"], "subfolder": img.get("subfolder", ""), "type": img.get("type", "output")})
    data = requests.get(f"{SERVER}/view?{q}", timeout=60).content
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    open(out_path, "wb").write(data)

    sec = round(time.time() - t0, 1)
    LOG.append({"pose": os.path.basename(skel_path).replace("_skeleton", ""), "prompt": prompt, "negative": negative,
                "use_pose": use_pose, "seed": seed, "size": f"{W}x{H}", "steps": STEPS, "cfg": CFG,
                "output": os.path.basename(out_path), "sec": sec})
    print(f"저장: {out_path}  ({W}x{H}, {sec}초)")
    return out_path

def 나란히(paths, titles=None, size=4):
    # 이미지 여러 장을 한 줄로 보여 준다
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size * 1.3), squeeze=False)
    for a, p, t in zip(axes[0], paths, titles or [os.path.basename(p) for p in paths]):
        a.imshow(Image.open(p)); a.set_title(t, fontsize=9); a.axis("off")
    plt.tight_layout(); plt.show()

print("준비 완료: 포즈로_생성(뼈대경로, 프롬프트, 저장경로)")

## [단계 8] 한 장 만들어 보기

**이 셀이 하는 일:** 참조 사진마다 결과 한 장씩을 만들어 `samples/output_NN.png` 로 저장하고, **원본 사진 · 뼈대 · 결과**를 나란히 보여 줍니다.
비교를 위해 **포즈 조건 없이 프롬프트만** 넣은 결과(`output_NN_nopose.png`)도 같은 시드로 하나 만듭니다.

- `PROMPT` 를 원하는 장면으로 바꿔 보십시오. 영어 문장이 잘 통합니다.
- 첫 장은 모델을 GPU에 올리느라 오래 걸리고, 그다음부터는 빨라집니다.

**확인할 것:** 결과의 팔·다리·고개 방향이 원본 사진과 같은가? 뼈대 선이 그림에 찍혀 나오지는 않았는가?

In [ ]:
# [단계 8] 참조 사진마다 같은 자세의 결과 1장 + 포즈 조건 없는 비교 1장
PROMPT = "A firefighter in a yellow uniform standing in a city street, photorealistic, natural daylight"
SEED = 42

for i, skel in enumerate(SKELETONS, start=1):
    out = f"{SAMPLES}/output_{i:02d}.png"
    nopose = f"{SAMPLES}/output_{i:02d}_nopose.png"
    포즈로_생성(skel, PROMPT, out, seed=SEED)
    포즈로_생성(skel, PROMPT, nopose, seed=SEED, use_pose=False)
    나란히([POSE_FILES[i-1], skel, out, nopose], ["reference photo", "skeleton (pose condition)", "result", "no pose condition"])

## [단계 9] 실험 A — 같은 포즈, 다른 프롬프트

**이 셀이 하는 일:** 포즈는 `pose_01` 하나로 고정하고, **프롬프트만** 바꿔 세 장을 만듭니다. 시드도 같게 둡니다.
결과는 `samples/expA_01.png` … 로 저장됩니다.

**볼 것:** 인물·옷·배경은 프롬프트대로 바뀌는데 **자세는 그대로 유지되는가?** 프롬프트 내용이 자세와 충돌할 때(예: 앉은 자세에 'running') 어느 쪽이 이기는가?

In [ ]:
# [단계 9] 실험 A: pose_01 고정, 프롬프트만 바꿔 3장
EXP_A_PROMPTS = [
    "An astronaut in a white spacesuit on the surface of the moon, photorealistic",
    "A chef in a white uniform in a busy restaurant kitchen, warm lighting, photorealistic",
    "A watercolor illustration of a young girl in a red dress in a flower garden",
]
skel = SKELETONS[0]
outs_a = []
for j, p in enumerate(EXP_A_PROMPTS, start=1):
    outs_a.append(포즈로_생성(skel, p, f"{SAMPLES}/expA_{j:02d}.png", seed=SEED))

나란히([skel] + outs_a, ["pose: pose_01"] + [f"A{j}: " + p[:38] + "…" for j, p in enumerate(EXP_A_PROMPTS, 1)])

## [단계 10] 실험 B — 다른 포즈, 같은 프롬프트

**이 셀이 하는 일:** 프롬프트와 시드는 고정하고, **포즈(참조 사진)만** 바꿔 만듭니다. 결과는 `samples/expB_NN.png` 로 저장됩니다.
참조 사진이 1장뿐이면 이 실험은 건너뜁니다. 단계 5로 돌아가 사진을 더 올리십시오.

**볼 것:** 포즈가 바뀌면 결과 자세도 따라 바뀌는가? 어떤 자세(앉기, 팔 교차, 옆모습 등)는 잘 안 따라오는가? 손·발끝 같은 작은 관절은 어떤가?

In [ ]:
# [단계 10] 실험 B: 프롬프트 고정, 포즈만 바꿔 생성
EXP_B_PROMPT = "A businessman in a navy suit in a modern office, photorealistic, soft window light"

if len(SKELETONS) < 2:
    print("참조 사진이 1장뿐입니다. 단계 5에서 자세가 다른 사진을 더 올린 뒤 단계 6부터 다시 실행하세요.")
else:
    outs_b = []
    for i, skel in enumerate(SKELETONS, start=1):
        outs_b.append(포즈로_생성(skel, EXP_B_PROMPT, f"{SAMPLES}/expB_{i:02d}.png", seed=SEED))
    나란히(SKELETONS + outs_b, [f"pose_{i:02d}" for i in range(1, len(SKELETONS) + 1)]
                              + [f"B{i}: pose_{i:02d}" for i in range(1, len(SKELETONS) + 1)], size=3)

## [단계 11] 기록 저장 + 결과 내려받기

**이 셀이 하는 일:**
1. 지금까지 만든 모든 이미지의 프롬프트·시드·설정을 표로 정리해 `prompts.md` 로 저장합니다. (재현용 기록)
2. `samples/` 폴더와 `prompts.md` 를 `pose-image-tool.zip` 하나로 묶어 내 컴퓨터로 내려받습니다.

내려받은 zip을 풀어 GitHub 저장소의 `samples/`, `prompts.md` 자리에 넣으면 됩니다.

In [ ]:
# [단계 11] prompts.md 저장 + samples/ 와 함께 zip 으로 내려받기
import shutil

lines = [
    "# 테스트에 쓴 프롬프트 모음", "",
    "`pose_tool.ipynb` 가 실제로 생성한 기록입니다. 모델: FLUX.2-klein 4B (GGUF Q4_K_M), ComfyUI, "
    f"{STEPS}스텝, cfg {CFG}, euler/simple.", "",
    "포즈 조건을 켠 생성에는 프롬프트 뒤에 아래 지시문이 자동으로 붙습니다.", "",
    "```", POSE_INSTRUCTION.strip(), "```", "",
    f"기본 네거티브: `{BASE_NEGATIVE}`", "",
    "| 결과 파일 | 포즈 | 포즈 조건 | 프롬프트 | 시드 | 크기 | 걸린 시간 |",
    "|---|---|---|---|---|---|---|",
]
for e in LOG:
    lines.append(f"| {e['output']} | {e['pose']} | {'켬' if e['use_pose'] else '끔'} | {e['prompt']} | "
                 f"{e['seed']} | {e['size']} | {e['sec']}초 |")

ROOT = "/content/pose-image-tool"
open(f"{ROOT}/prompts.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))

# samples/ 와 prompts.md 를 zip 하나로 묶는다
shutil.make_archive("/content/pose-image-tool", "zip", ROOT)
from google.colab import files
files.download("/content/pose-image-tool.zip")

## (선택) 부록 B — 진짜 ControlNet으로 비교하기: SD 1.5 + OpenPose ControlNet

**이 셀이 하는 일:** 같은 뼈대 그림을 **ControlNet**(생성 모델 옆에 붙어 자세를 강하게 고정하는 보조 신경망)으로 넣어 봅니다.
FLUX.2-klein의 '참조 이미지' 방식과 **자세를 얼마나 정확히 따르는지** 비교하는 용도입니다.

- 모델: `stable-diffusion-v1-5` + `lllyasviel/control_v11p_sd15_openpose` (라이선스 CreativeML OpenRAIL-M)
- 메모리를 비우려고 **ComfyUI 서버를 먼저 끕니다.** 다시 FLUX로 생성하려면 단계 4부터 다시 실행하십시오.
- 결과는 `samples/controlnet_NN.png` 로 저장됩니다. 저장 뒤 단계 11을 다시 실행하면 zip에 함께 들어갑니다.

꼭 필요한 셀은 아닙니다. klein 결과가 자세를 잘 못 따라올 때, 그 차이를 관찰 기록에 적는 근거로 쓰십시오.

In [ ]:
# [부록 B] SD 1.5 + OpenPose ControlNet 으로 같은 뼈대에서 생성 (비교용)
!pkill -f "main.py" 2>/dev/null
import torch, gc
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
gc.collect(); torch.cuda.empty_cache()

controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_openpose", torch_dtype=torch.float16)
sd = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", controlnet=controlnet,
    torch_dtype=torch.float16, safety_checker=None).to("cuda")
sd.scheduler = UniPCMultistepScheduler.from_config(sd.scheduler.config)

def 컨트롤넷_생성(skel_path, prompt, out_path, seed=42):
    # 뼈대 그림을 ControlNet 조건으로 넣어 생성 (SD1.5 는 긴 변 768 정도가 알맞다)
    w, h = Image.open(skel_path).size
    s = 768 / max(w, h)
    W, H = round(w * s / 8) * 8, round(h * s / 8) * 8
    skel = Image.open(skel_path).convert("RGB").resize((W, H))
    img = sd(prompt, image=skel, num_inference_steps=25, width=W, height=H,
             negative_prompt="lowres, bad anatomy, extra limbs, deformed hands",
             generator=torch.Generator("cuda").manual_seed(seed)).images[0]
    img.save(out_path)
    LOG.append({"pose": os.path.basename(skel_path).replace("_skeleton", ""), "prompt": prompt + " [SD1.5+ControlNet]",
                "negative": "", "use_pose": True, "seed": seed, "size": f"{W}x{H}", "steps": 25, "cfg": 7.5,
                "output": os.path.basename(out_path), "sec": "-"})
    return out_path

for i, skel in enumerate(SKELETONS, start=1):
    out = 컨트롤넷_생성(skel, PROMPT, f"{SAMPLES}/controlnet_{i:02d}.png", seed=SEED)
    나란히([POSE_FILES[i-1], skel, f"{SAMPLES}/output_{i:02d}.png", out],
           ["reference photo", "skeleton", "FLUX.2-klein (reference)", "SD1.5 + ControlNet"])

## [단계 12] 관찰 기록 — 무엇을 바꿔 보았고, 어떻게 달라졌는가

아래는 이 노트를 Colab 무료 T4에서 직접 실행한 결과입니다. (2026-11-13, 시드 42 고정, 4스텝, cfg 4.0)
참조 사진 세 장은 Gemini로 만든 가상 인물 이미지입니다: `pose_01` 서서 오른팔 들기·왼손 허리, `pose_02` 의자에 옆으로 앉기, `pose_03` 요가 전사 자세.

### 기본 동작 (단계 8) — 포즈 조건 켬 vs 끔
| | pose_01 손 들기 | pose_02 앉기 | pose_03 전사 자세 |
|---|---|---|---|
| 참조 사진 | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/pose_01.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/pose_02.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/pose_03.png" width="120"> |
| 뼈대 | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/pose_01_skeleton.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/pose_02_skeleton.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/pose_03_skeleton.png" width="120"> |
| 포즈 조건 켬 | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/output_01.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/output_02.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/output_03.png" width="120"> |
| 포즈 조건 끔 | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/output_01_nopose.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/output_02_nopose.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/output_03_nopose.png" width="120"> |

- **포즈 조건을 끄면** 프롬프트·시드가 같으니 세 장이 똑같이 차렷 자세로 나왔습니다. **켜면** 세 장이 각각 참조 사진의 자세를 따라갔습니다. 결과를 바꾼 것은 뼈대 하나뿐이라는 뜻입니다.
- **맞은 곳:** pose_01의 든 팔 방향과 높이, pose_03의 양팔 수평·넓게 벌린 다리·굽힌 무릎 방향은 거의 그대로 옮겨졌습니다.
- **어긋난 곳:** pose_01에서 허리를 짚은 왼손이 가슴·배 앞으로 올라왔습니다. pose_02는 의자가 없는 채로 무릎만 굽혀 엉거주춤 걷는 듯한 자세가 됐습니다.
- **뼈대 선이 찍혀 나오지는 않았습니다.** 네거티브와 지시문("뼈대를 그리지 말 것")이 통했습니다.
- **속도:** 첫 장은 모델을 올리느라 95.6초, 그다음부터 포즈 조건을 켜면 약 37~39초, 끄면 약 19~20초였습니다. 참조 이미지가 입력에 더해져 계산이 두 배쯤 늘어납니다.

### 실험 A — 같은 포즈(pose_01), 프롬프트만 바꿈
| 뼈대 | 우주비행사 | 요리사 | 수채화 소녀 |
|---|---|---|---|
| <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/pose_01_skeleton.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/expA_01.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/expA_02.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/expA_03.png" width="120"> |

- **바뀐 것:** 인물, 옷, 배경, 화풍(사진 → 수채화)까지 프롬프트대로 완전히 바뀌었습니다.
- **유지된 것:** 오른팔을 높이 든 자세는 세 장 모두 유지됐습니다.
- **잘 안 따라온 것:** 손 모양입니다. 참조는 손바닥을 편 채 흔드는 손인데 우주비행사·요리사는 주먹을 쥐었습니다. 왼손 허리 짚기도 수채화 소녀만 따라 했고 나머지는 가슴·배 앞에 손을 두었습니다. 뼈대의 손 관절 점이 작고 촘촘해서 큰 팔 방향보다 약하게 전달되는 것으로 보입니다.

### 실험 B — 같은 프롬프트(정장 차림 회사원), 포즈만 바꿈
| pose_01 | pose_02 | pose_03 |
|---|---|---|
| <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/expB_01.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/expB_02.png" width="120"> | <img src="https://raw.githubusercontent.com/k-im-rose/pose-image-tool/main/samples/expB_03.png" width="120"> |

- **바뀐 것:** 몸 자세만 바뀌었습니다. 인물·옷·사무실 배경은 세 장이 거의 같습니다.
- **잘 따라온 자세:** pose_03(전사 자세)이 가장 정확했습니다. 팔 높이, 다리 벌린 폭, 무릎 방향, 옆으로 돌린 고개까지 맞았습니다.
- **잘 안 따라온 자세:** pose_02(앉기)는 자세는 앉은 모양인데 **의자가 사라져 공중에 앉은** 그림이 됐습니다. 뼈대에는 사람 관절만 있고 의자 정보가 없으며, 프롬프트에도 의자가 없었기 때문입니다. 옆모습이라 반대쪽 팔·다리가 가려진 것도 뼈대가 불완전한 이유입니다.

### 어느 쪽을 바꿨을 때 결과가 크게 달라졌나
- **역할이 나뉩니다.** 프롬프트는 "누가·어디서·어떤 결로"를, 뼈대는 "어떤 자세로"를 정합니다. 프롬프트를 바꾸면 그림 전체 인상이 크게 바뀌지만 자세는 그대로이고, 포즈를 바꾸면 인물·배경은 그대로인데 몸의 모양이 크게 바뀝니다.
- 다만 **둘이 부딪히면 둘 다 반쯤만 이깁니다.** 단계 8의 프롬프트에 standing(서 있는)이 들어 있었는데 앉은 뼈대를 넣자, 서지도 앉지도 않은 엉거주춤한 자세가 나왔습니다.

### 한계와 다음에 해 볼 것
- **물건은 뼈대에 안 담깁니다.** 앉기·기대기처럼 물건이 필요한 자세는 프롬프트에 "sitting on a wooden chair"처럼 물건을 직접 적어야 합니다.
- **손 모양·손가락은 약하게 따라옵니다.** 손이 중요하면 손이 크게 나온 참조 사진을 쓰거나, 부록 B의 ControlNet 방식과 비교해 볼 만합니다.
- **옆모습은 가려진 관절이 빠집니다.** 정면이나 3/4 각도 사진이 더 안정적입니다.
- **OpenPose가 CPU에서 돌아** 사진 세 장 뼈대 추출에 약 2분이 걸렸습니다. 사진이 많으면 GPU로 옮기는 것을 고려합니다.